In [0]:
%pip install langchain
%pip install databricks-vectorsearch
dbutils.library.restartPython()

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Not uninstalling langchain-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5b77fe90-9b03-43ff-afd6-fc96d76e5212
    Can't uninstall 'langchain-core'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-5b77fe90-9b03-43ff-afd6-fc96d76e5212
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at 

In [0]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from databricks.vector_search.client import VectorSearchClient

load_dotenv()


True

In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    index = VectorSearchClient(disable_notice=True).get_index(
        endpoint_name="25280065_vector_endpoint",
        index_name="pa3_catalog.vector_index.fixed_vector_index"
    )
    results = index.similarity_search(
        query_text=question,
        columns=["chunk_text"],
        num_results=topk
    )
    rows = results.get("result", {}).get("data_array", [])
    raw_docs = [row[0] for row in rows if row and len(row) > 0]
    return "\n\n---\n\n".join(raw_docs)


@tool
def vector_search_tool(query: str) -> str:
    """
    Searches the Databricks Vector Search index for document chunks relevant 
    to the query. Use this tool first when answering any question about 
    course content, documentation, or SearchAgent-X. Returns the top 3 
    most semantically similar chunks as formatted text.
    """
    try:
        context = retrieve_formatted_context(query)
        if not context.strip():
            return "No relevant documents found in the vector store."
        return context
    except Exception as e:
        return f"Database search failed: {e}"

@tool
def read_fallback_document(category: str) -> str:
    """
    Reads a pre-stored text document for a given category when the vector 
    store does not contain relevant information. Use this tool as a fallback 
    when vector_search_tool returns no useful results. 
    Supported categories: 'databricks', 'azure'.
    """
    file_map = {
        "databricks": "databricks_info.txt",
        "azure": "azure_info.txt"
    }
    category = category.strip().lower()
    if "databricks" in category:
        category = "databricks"
    elif "azure" in category:
        category = "azure"
    
    file_path = file_map.get(category)
    if not file_path:
        return f"No documentation found for category: {category}. Supported: 'databricks', 'azure'."
    try:
        with open(file_path, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error reading document: {e}"
    

def build_stateful_mcp_agent():
    llm = AzureChatOpenAI(
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
        azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"),
        api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
        api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
        temperature=0.0,
    )

    tools = [vector_search_tool, read_fallback_document] 
    system_prompt = (
        "You are a precise data engineering assistant with memory. "
        "Always use vector_search_tool first to answer questions about documentation or SearchAgent-X. "
        "If vector_search_tool returns no useful results, use read_fallback_document with "
        "category 'azure' or 'databricks' as appropriate. "
        "Remember previous answers in this conversation to answer follow-up questions."
    )
    memory = MemorySaver()
    mcp_agent = create_react_agent(
        model=llm,
        tools=tools,
        checkpointer=memory,
        prompt=system_prompt,
    )
    return mcp_agent

agent = build_stateful_mcp_agent()

/home/spark-5b77fe90-9b03-43ff-afd6-fc/.ipykernel/3148/command-7902319441856129-3402289290:78: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mcp_agent = create_react_agent(


In [0]:
# Summary: Executes queries against the agent while maintaining a consistent session thread.
def execute_mcp_queries(mcp_agent, queries: list, thread_id: str):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }    
    for query in queries:
        print(f"\nUser: {query}")
        payload = {
            "messages": [{"role": "user", "content": query}]
        }        
        response = mcp_agent.invoke(payload, config=config)
        final_message = response["messages"][-1].content
        
        print(f"Agent: {final_message}")

In [0]:
validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?"
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

--- TESTING STATEFUL MCP AGENT ---

User: What tools do you have?
Agent: I have access to the following tools:

1. vector_search_tool: Searches the Databricks Vector Search index for document chunks relevant to a query. It returns the top 3 most semantically similar chunks. This is useful for answering questions about course content, documentation, or SearchAgent-X.

2. read_fallback_document: Reads a pre-stored text document for a given category ('databricks' or 'azure') when the vector search does not return useful information. This serves as a fallback option.

3. multi_tool_use.parallel: Allows running multiple functions tools in parallel if needed.

If you have any questions or need information, I can use these tools to assist you!

User: What is searchAgent-X?
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Agent: Search